In [1]:
# General notebook settings
import logging
import warnings

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
# pandas<3.0.3 sets the `locs` attribute deprecated in matplotlib>=3.11
warnings.filterwarnings("ignore", message="The locs attribute was deprecated")
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False


# Quickstart 3 - Investments & Storage


## Problem Description

A data centre in Seville, Spain, has a constant demand of 100 MW. The operator
considers investing in on-site solar PV and battery storage to reduce reliance
on grid electricity, which is priced at 120 €/MWh. The investment costs and
characteristics of the components are as follows:


| Component | Overnight Cost | Lifetime | Discount Rate |
|-----------|----------------|----------|---------------|
| Solar PV  | 400 €/kW       | 25 years | 5%            |
| Battery Storage   | 150 €/kWh      | 25 years | 5%            |
| Battery Inverter  | 170 €/kW       | 10 years | 5%            |

- The battery storage system has a round-trip efficiency of 90% and an energy-to-power ratio of 4 hours.
- The solar PV plant has a capacity factor time series given [here](https://model.energy/data/time-series-f17c3736a2719ce7da58484180d89e2d.csv).
- Assume that feeding electricity into the grid is not allowed.

Find the least-cost investment in solar PV and battery storage to cover the load. What is the average cost per unit of electricity consumed?
How much electricity is consumed from the grid and when? How is the battery operated?

## PyPSA Solution

We start by creating a new network with a single bus and the data centre load.
We also add a generator to model supply from the grid priced at 120 €/MWh.
These are the fixed components of the network.

In [2]:
import numpy as np
import pandas as pd

import pypsa
from pypsa.costs import annuity

n = pypsa.Network()

n.add("Bus", "seville")

n.add("Load", "demand", bus="seville", p_set=100)

n.add("Generator", "grid", bus="seville", p_nom=100, marginal_cost=120, carrier="grid");

/home/runner/work/PyPSA/PyPSA/pypsa/network/io.py:2085: FutureWarning: pandas infers the `str` dtype for string data since its version 3.0. PyPSA still converts it back to numpy object dtype on import, but will keep it from PyPSA 2.0 on. Set `pypsa.options.api.legacy_string_dtype` explicitly to suppress this warning.
  new_static = _coerce_string_dtypes(new_static)


Next, we read in the capacity factor time series of the network, which covers hourly data for the year 2011.

In [3]:
p_max_pu = pd.read_csv(
    "https://model.energy/data/time-series-f17c3736a2719ce7da58484180d89e2d.csv",
    index_col=0,
    parse_dates=True,
)["solar"]
p_max_pu[7:15]

time
2011-01-01 07:00:00    0.000
2011-01-01 08:00:00    0.000
2011-01-01 09:00:00    0.048
2011-01-01 10:00:00    0.118
2011-01-01 11:00:00    0.214
2011-01-01 12:00:00    0.243
2011-01-01 13:00:00    0.219
2011-01-01 14:00:00    0.138
Name: solar, dtype: float64

We need to tell PyPSA that these are the snapshots (time steps) we want to optimise over.

In [4]:
n.set_snapshots(p_max_pu.index)
len(n.snapshots)

8760

Then, we add the solar PV with the availability time series as `p_max_pu`, the annualised costs in €/MW/a as `capital_cost` and mark the component as extendable with `p_nom_extendable`.

In [5]:
n.add(
    "Generator",
    "solar",
    bus="seville",
    p_max_pu=p_max_pu,
    capital_cost=annuity(0.05, 25) * 400_000,
    p_nom_extendable=True,
    carrier="solar",
);

Similarly, we add the battery storage. Here, we need to take extra care with the multiple cost components of the battery system for the `capital_cost`, which is given per MWh of energy capacity for stores, and the energy-to-power ratio (`max_hours`).

In [6]:
cc_inverter = annuity(0.05, 10) * 170_000
cc_storage = annuity(0.05, 25) * 150_000

n.add(
    "Store",
    "battery",
    bus="seville",
    capital_cost=cc_inverter / 4 + cc_storage,
    e_nom_extendable=True,
    carrier="battery",
    efficiency_store=np.sqrt(0.9),
    efficiency_dispatch=np.sqrt(0.9),
    max_hours=4,
);

Now, the model is ready to be solved:

In [7]:
n.optimize()

/tmp/ipykernel_2936/1261279110.py:1: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  n.optimize()
Index(['seville'], dtype='object', name='name')


Index(['grid', 'solar'], dtype='object', name='name')


Index(['battery'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Gurobi solver


INFO:linopy.model:Solver options:
 - log_to_console: False


INFO:linopy.io:Writing objective.


... (12 outputs left out)

Set parameter LogToConsole to value 0


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 43802 primals, 105122 duals
Objective: 5.17e+07
Solver: gurobi
Runtime: 0.64s
Dual bound: 5.17e+07
Solver model: available
Solver message: 2



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, Store-ext-e-lower, Store-ext-e-upper, Store-ext-p-lower, Store-ext-p-upper, Store-ext-p_store-lower, Store-ext-p_store-upper, Store-energy_balance were not assigned to the network.


('ok', 'optimal')

To retrieve the optimised capacities, we can either directly access the `p_nom_opt` attribute of the components (`e_nom_opt` in MWh for the battery store), or use the `n.statistics` module.

In [8]:
display(n.generators.p_nom_opt)
display(n.stores.e_nom_opt)

name
grid     100.000000
solar    656.631357
Name: p_nom_opt, dtype: float64

name
battery    1380.212844
Name: e_nom_opt, dtype: float64

In [9]:
n.statistics.optimal_capacity()

component  carrier
Generator  grid        100.00000
           solar       656.63136
Store      battery    1380.21284
dtype: float64

The statistics module also provides a convenient way to calculate investment and operational costs, as well as the average cost per unit of electricity consumed.

In [10]:
totex = {"opex": n.statistics.opex(), "capex": n.statistics.capex()}
pd.concat(totex, axis=1).div(1e6).round(2)  # M€/a

opex  capex
component carrier              
Generator grid     10.82    NaN
          solar      NaN  18.64
Store     battery    NaN  22.29

In [11]:
(n.statistics.capex().sum() + n.statistics.opex().sum()) / 100 / 8760  # €/MWh

np.float64(59.07075941164384)

The statistics module can also give you the energy balances of the system, to see how much electricity is consumed from the grid and when, and what the battery storage losses are.

In [12]:
n.statistics.energy_balance().div(1e3)  # GWh

component  carrier  bus_carrier
Generator  grid     AC              90.200860
           solar    AC             836.119621
Load       -        AC            -876.000000
Store      battery  AC             -50.320481
dtype: float64

To access and plot the state of charge profile of the battery for January, run

In [13]:
n.stores_t.e.loc["2011-01"].plot(backend="plotly")

The statistics functions also have built-in plotting capabilities, e.g. to plot the dispatch profiles of the system as stacked area charts.

In [14]:
n.add(
    "Carrier",
    ["grid", "solar", "battery", "AC"],
    color=["blue", "yellow", "green", "k"],
)

n.statistics.energy_balance.iplot()

Finally, any network object can be exported to files, for example to Excel or NetCDF, for further analysis or reporting.  Importing from files is of course also possible.

In [15]:
n.export_to_excel("data-centre-investment.xlsx")
n.export_to_netcdf("data-centre-investment.nc")

o = pypsa.Network("data-centre-investment.nc")

INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'data-centre-investment.xlsx contains: carriers, generators, loads, stores, buses


INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'data-centre-investment.nc contains: carriers, generators, loads, stores, buses


INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, carriers, generators, loads, stores


Find many more extensive examples in the [examples](https://docs.pypsa.org/latest/examples/examples/) section.

The [user guide](https://docs.pypsa.org/latest/user-guide/user-guide/) section contains detailed information on architecture, components, problem formulation and utilities.